# Security & Governance for Shared Data

Deep dive into RBAC, row access policies, masking policies, database roles, and auditing for data sharing.

**Key concepts covered:**
- Provider & consumer role separation (`CREATE SHARE`, `IMPORT SHARE`)
- Secure views — the recommended sharing primitive
- Database roles — granular consumer access within a single share
- `IS_DATABASE_ROLE_IN_SESSION()` — the correct function for policies in shared context (`CURRENT_ROLE()` returns `NULL` in consumer accounts)
- Row access policies & masking policies on shared data (Enterprise Edition+)
- `SIMULATED_DATA_SHARING_CONSUMER` — test shared views without a real consumer
- `ACCESS_HISTORY` & `QUERY_HISTORY` for audit

**References:**
- [Share data protected by a policy](https://docs.snowflake.com/en/user-guide/data-sharing-policy-protected-data)
- [IS_DATABASE_ROLE_IN_SESSION](https://docs.snowflake.com/en/sql-reference/functions/is_database_role_in_session)
- [Use secure objects to control data access](https://docs.snowflake.com/en/user-guide/data-sharing-secure-views)
- [GRANT DATABASE ROLE … TO SHARE](https://docs.snowflake.com/en/sql-reference/sql/grant-database-role-share)
- [ACCESS_HISTORY view](https://docs.snowflake.com/en/sql-reference/account-usage/access_history)

## 1. Setup

In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE DATABASE sec_sharing_demo;
CREATE SCHEMA sec_sharing_demo.raw;
CREATE SCHEMA sec_sharing_demo.shared;

-- Sample data
CREATE TABLE sec_sharing_demo.raw.employees AS
SELECT
    ROW_NUMBER() OVER (ORDER BY c_custkey) AS emp_id,
    c_name AS emp_name,
    'emp_' || c_custkey || '@example.com' AS email,
    CASE MOD(c_custkey, 3)
        WHEN 0 THEN 'Engineering'
        WHEN 1 THEN 'Sales'
        ELSE 'Finance'
    END AS department,
    (50000 + MOD(c_custkey * 137, 100000))::NUMBER(10,2) AS salary,
    c_mktsegment AS region
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
LIMIT 1000;

## 2. RBAC — Provider & Consumer Roles

- **`share_admin`** — can create shares, manage shared objects, create secure views
- **`share_monitor`** — read-only access to `SNOWFLAKE.ACCOUNT_USAGE` for auditing

> `CREATE SHARE` privilege is account-level (was split from `MANAGE SHARE TARGET` in BCR 2024_07).

In [ ]:
USE ROLE SECURITYADMIN;

CREATE ROLE IF NOT EXISTS share_admin;
CREATE ROLE IF NOT EXISTS share_monitor;
GRANT ROLE share_admin TO ROLE SYSADMIN;
GRANT ROLE share_monitor TO ROLE share_admin;

GRANT CREATE SHARE ON ACCOUNT TO ROLE share_admin;
GRANT USAGE ON DATABASE sec_sharing_demo TO ROLE share_admin;
GRANT USAGE ON ALL SCHEMAS IN DATABASE sec_sharing_demo TO ROLE share_admin;
GRANT SELECT ON ALL TABLES IN SCHEMA sec_sharing_demo.raw TO ROLE share_admin;
GRANT CREATE VIEW ON SCHEMA sec_sharing_demo.shared TO ROLE share_admin;
GRANT IMPORTED PRIVILEGES ON DATABASE snowflake TO ROLE share_monitor;

## 3. Secure Views with Account-Based Filtering

Snowflake **strongly recommends** sharing secure views (not raw tables).  
- `SECURE_OBJECTS_ONLY = TRUE` is the default on `CREATE SHARE`.
- The optimizer does not expose the view definition to consumers.

In [ ]:
USE ROLE share_admin;
USE DATABASE sec_sharing_demo;

CREATE OR REPLACE SECURE VIEW shared.v_employees AS
SELECT emp_id, emp_name, department, region
FROM raw.employees;

CREATE OR REPLACE SECURE VIEW shared.v_department_summary AS
SELECT department, COUNT(*) AS headcount, AVG(salary) AS avg_salary
FROM raw.employees
GROUP BY department;

## 4. Database Roles for Granular Access

Database roles allow different consumers to see different subsets within the same share.

- `basic_reader` — access to `v_employees` only
- `analytics_reader` — access to all views in the `shared` schema

> **Note:** Future grants on database roles granted to shares are **not supported**.

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE DATABASE ROLE IF NOT EXISTS sec_sharing_demo.basic_reader;
CREATE DATABASE ROLE IF NOT EXISTS sec_sharing_demo.analytics_reader;

GRANT USAGE ON SCHEMA sec_sharing_demo.shared TO DATABASE ROLE sec_sharing_demo.basic_reader;
GRANT SELECT ON VIEW sec_sharing_demo.shared.v_employees TO DATABASE ROLE sec_sharing_demo.basic_reader;

GRANT USAGE ON SCHEMA sec_sharing_demo.shared TO DATABASE ROLE sec_sharing_demo.analytics_reader;
GRANT SELECT ON ALL VIEWS IN SCHEMA sec_sharing_demo.shared TO DATABASE ROLE sec_sharing_demo.analytics_reader;

## 5. Row Access Policy

Uses `IS_DATABASE_ROLE_IN_SESSION()` — **`CURRENT_ROLE()` returns `NULL`** in a consumer context.

- The function evaluates against the database containing the **protected table** (not the session database).
- Mapping tables must be in the **same database** as the protected table.

> Enterprise Edition+ required.

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE ROW ACCESS POLICY sec_sharing_demo.shared.rap_department
AS (dept STRING) RETURNS BOOLEAN ->
    IS_DATABASE_ROLE_IN_SESSION('SEC_SHARING_DEMO.ANALYTICS_READER')
    OR dept = 'Engineering';

ALTER VIEW sec_sharing_demo.shared.v_employees
    ADD ROW ACCESS POLICY sec_sharing_demo.shared.rap_department ON (department);

## 6. Masking Policy

Mask sensitive data based on database role context.

- `ANALYTICS_READER` sees full email; everyone else sees `*****@domain.com`.
- The database role must be in the **same database** as the protected table.

> Enterprise Edition+ required.

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE MASKING POLICY sec_sharing_demo.shared.mask_email
AS (val STRING) RETURNS STRING ->
    CASE
        WHEN IS_DATABASE_ROLE_IN_SESSION('SEC_SHARING_DEMO.ANALYTICS_READER') THEN val
        ELSE REGEXP_REPLACE(val, '.+\\@', '*****@')
    END;

-- Apply to a new view that includes email
CREATE OR REPLACE SECURE VIEW sec_sharing_demo.shared.v_employees_detail AS
SELECT emp_id, emp_name, email, department, region
FROM sec_sharing_demo.raw.employees;

ALTER VIEW sec_sharing_demo.shared.v_employees_detail
    MODIFY COLUMN email SET MASKING POLICY sec_sharing_demo.shared.mask_email;

## 7. Create Share with Database Roles

Grant database roles to the share so consumers can map them to their own account roles.

In [ ]:
USE ROLE share_admin;

CREATE OR REPLACE SHARE sec_governance_demo_share
    COMMENT = 'Demo: security & governance patterns';

GRANT USAGE ON DATABASE sec_sharing_demo TO SHARE sec_governance_demo_share;
GRANT USAGE ON SCHEMA sec_sharing_demo.shared TO SHARE sec_governance_demo_share;
GRANT SELECT ON ALL VIEWS IN SCHEMA sec_sharing_demo.shared TO SHARE sec_governance_demo_share;

USE ROLE ACCOUNTADMIN;
GRANT DATABASE ROLE sec_sharing_demo.basic_reader TO SHARE sec_governance_demo_share;
GRANT DATABASE ROLE sec_sharing_demo.analytics_reader TO SHARE sec_governance_demo_share;

## 8. Test with Simulated Consumer

The `SIMULATED_DATA_SHARING_CONSUMER` session parameter lets you preview how shared views behave for a consumer account **without needing one**.

- The executing role must **own** the view (and any nested views).

In [ ]:
USE ROLE ACCOUNTADMIN;
-- ALTER SESSION SET SIMULATED_DATA_SHARING_CONSUMER = '<ORG>.<ACCOUNT>';
-- Test basic_reader view
SELECT * FROM sec_sharing_demo.shared.v_employees LIMIT 10;
-- Test masked email view
SELECT * FROM sec_sharing_demo.shared.v_employees_detail LIMIT 10;
-- ALTER SESSION UNSET SIMULATED_DATA_SHARING_CONSUMER;

## 9. Audit & Access History

- `QUERY_HISTORY` — captures share-related DDL (CREATE, ALTER, GRANT, REVOKE).
- `ACCESS_HISTORY` (Enterprise+) — tracks `direct_objects_accessed`, `base_objects_accessed`, and `policies_referenced`.
- Provider queries are **not visible** to consumers; consumer queries are **not visible** to providers.
- `ACCESS_HISTORY` latency: up to 3 hours.

In [ ]:
USE ROLE share_monitor;

-- Recent DDL changes to shares
SELECT query_start_time, user_name, query_text
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE query_text ILIKE '%SHARE%'
  AND query_type IN ('CREATE', 'ALTER', 'GRANT', 'REVOKE')
  AND start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY query_start_time DESC
LIMIT 20;

-- Access history on shared objects (Enterprise+)
SELECT
    query_start_time,
    user_name,
    direct_objects_accessed
FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY
WHERE ARRAY_SIZE(direct_objects_accessed) > 0
  AND query_start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
ORDER BY query_start_time DESC
LIMIT 20;

## 10. Teardown

In [ ]:
USE ROLE ACCOUNTADMIN;
DROP SHARE IF EXISTS sec_governance_demo_share;
DROP DATABASE IF EXISTS sec_sharing_demo;
DROP ROLE IF EXISTS share_admin;
DROP ROLE IF EXISTS share_monitor;

## References

| Topic | Link |
|-------|------|
| Share data protected by a policy | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/data-sharing-policy-protected-data) |
| IS_DATABASE_ROLE_IN_SESSION | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/functions/is_database_role_in_session) |
| Secure views & sharing | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/data-sharing-secure-views) |
| GRANT DATABASE ROLE … TO SHARE | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/sql/grant-database-role-share) |
| ACCESS_HISTORY view | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/account-usage/access_history) |
| Row access policies | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/security-row-intro) |
| Column-level security (masking) | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/security-column-intro) |
| Enable non-ACCOUNTADMIN sharing roles | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/security-access-privileges-shares) |
| SIMULATED_DATA_SHARING_CONSUMER | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/sql/alter-session) |